In [2]:
from dotenv import load_dotenv
load_dotenv("/home/radha/Desktop/scfp125033_mca_lab/myenv/.env")

True

In [3]:
import os

In [4]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0
)

print("Groq LLM configured successfully!")

Groq LLM configured successfully!


In [6]:
class ResearchData(BaseModel):
    research_question: str = Field(description="The main question the research investigates")
    method: str = Field(description="The method or approach used in the research")
    key_finding: str = Field(description="The most important finding of the research")

In [7]:
parser = PydanticOutputParser(pydantic_object=ResearchData)
print("Research output structure created successfully!")

Research output structure created successfully!


In [8]:
abstract_text = """
This study investigates whether regular physical activity improves sleep quality
among university students. Researchers conducted a 12-week study involving 200
students. Half of the participants followed a regular exercise program, while the
other half continued their usual routines. Sleep quality was measured using
questionnaires before and after the study. The results showed that students in
the exercise group reported better sleep quality than those in the comparison
group. The findings suggest that regular physical activity may help university
students sleep better.
"""

In [9]:
print(abstract_text)


This study investigates whether regular physical activity improves sleep quality
among university students. Researchers conducted a 12-week study involving 200
students. Half of the participants followed a regular exercise program, while the
other half continued their usual routines. Sleep quality was measured using
questionnaires before and after the study. The results showed that students in
the exercise group reported better sleep quality than those in the comparison
group. The findings suggest that regular physical activity may help university
students sleep better.



In [10]:
extract_prompt = PromptTemplate(
    template="""Read the research abstract and extract the following information:
1. Research question
2. Method
3. Key finding

Use only information stated in the abstract.
Do not invent details.

Abstract:{abstract_text}
{format_instructions}""",
    input_variables=["abstract_text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

In [11]:
extract_chain = extract_prompt | llm | parser

In [12]:
structured_data = extract_chain.invoke({
    "abstract_text": abstract_text
})

In [13]:
print(structured_data.model_dump_json(indent=2))

{
  "research_question": "Does regular physical activity improve sleep quality among university students?",
  "method": "A 12‑week study with 200 students, half of whom followed a regular exercise program and half who continued their usual routines; sleep quality was measured using questionnaires before and after the study.",
  "key_finding": "Students in the exercise group reported better sleep quality than those in the comparison group."
}


In [14]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a science communicator explaining research to a general audience. "
        "Use simple, everyday language. Avoid technical jargon. "
        "Explain the main question, what researchers did, and what they found. "
        "Use only the provided structured information. Do not add unsupported claims."
    ),
    ("human","""Research question: {research_question}

Method: {method}

Key finding: {key_finding}

Write a short, clear summary for someone without a scientific background.
"""
    )
])

In [15]:
summary_chain = summary_prompt | llm | StrOutputParser()

In [16]:
layperson_summary = summary_chain.invoke(
    structured_data.model_dump()
)

In [17]:
print("LAYPERSON SUMMARY")
print("-----------------")
print(layperson_summary)

LAYPERSON SUMMARY
-----------------
**Short Summary**

**What the study asked:**  
Do students who exercise regularly sleep better than those who don’t?

**How the researchers tested it:**  
- 200 university students were split into two groups.  
- One group (100 students) followed a regular exercise program for 12 weeks.  
- The other group (100 students) kept doing whatever they normally did.  
- Both groups answered sleep‑quality questionnaires before the study started and again after 12 weeks.

**What they found:**  
Students who exercised reported noticeably better sleep quality than the students who didn’t change their routine.  

In plain terms, regular physical activity seems to help university students sleep better.


In [18]:
print("STEP 1: STRUCTURED EXTRACTION")
print(structured_data.model_dump_json(indent=2))

print("\nSTEP 2: LAYPERSON SUMMARY")
print(layperson_summary)

STEP 1: STRUCTURED EXTRACTION
{
  "research_question": "Does regular physical activity improve sleep quality among university students?",
  "method": "A 12‑week study with 200 students, half of whom followed a regular exercise program and half who continued their usual routines; sleep quality was measured using questionnaires before and after the study.",
  "key_finding": "Students in the exercise group reported better sleep quality than those in the comparison group."
}

STEP 2: LAYPERSON SUMMARY
**Short Summary**

**What the study asked:**  
Do students who exercise regularly sleep better than those who don’t?

**How the researchers tested it:**  
- 200 university students were split into two groups.  
- One group (100 students) followed a regular exercise program for 12 weeks.  
- The other group (100 students) kept doing whatever they normally did.  
- Both groups answered sleep‑quality questionnaires before the study started and again after 12 weeks.

**What they found:**  
Studen